# Lab 2 · Extracción con esquema y citas verificadas

**Sesión Norma+ · Diplomado de Ciencia de Datos e IA para Ciencias Sociales**

En el Lab 1 aislamos la zona argumentativa y la partimos en chunks. Ahora le pedimos a Claude que extraiga **criterios jurídicos** de cada chunk. Tres ideas guían el lab:

1. **Esquema antes que prompt.** Definimos con Pydantic qué forma debe tener la respuesta y obligamos al modelo a usarla.
2. **Una cita que no está en el texto no existe.** Verificamos que cada cita sea literal y la anclamos a su página.
3. **Fallar cerrado.** Lo que no pasa una verificación se marca y se aparta; nunca se «arregla» en silencio.

Modelo: `claude-haiku-4-5`. Costo esperado de todo el lab: menos de USD 0.10.

## 0 · Preparación

In [ ]:
!pip install -q anthropic pydantic pdfplumber pandas

In [ ]:
# Descarga del dataset desde el repositorio público del curso (git viene instalado en Colab).
import os, shutil, zipfile
REPO_URL = "https://github.com/benjov/laboratorio-norma-notebooks"

if not os.path.exists("norma_mini_dataset"):
    !git clone -q --depth 1 "$REPO_URL" repo_curso
    if os.path.exists("repo_curso/norma_mini_dataset"):
        shutil.copytree("repo_curso/norma_mini_dataset", "norma_mini_dataset")
    else:
        print("No se pudo descargar de GitHub. Sube norma_mini_dataset.zip a mano:")
        from google.colab import files
        files.upload()
        zipfile.ZipFile("norma_mini_dataset.zip").extractall(".")
print(sorted(os.listdir("norma_mini_dataset")))

In [ ]:
from google.colab import userdata
import anthropic
client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))

In [ ]:
# Guardián de presupuesto: envuelve al cliente, suma el costo de cada llamada y se detiene al llegar al límite.
# Precios en USD por millón de tokens (entrada, salida). Un modelo no listado usa un precio conservador.
PRECIOS = {"claude-haiku-4-5": (1.0, 5.0)}
PRECIO_CONSERVADOR = (5.0, 25.0)

class Presupuesto:
    def __init__(self, client, limite_usd):
        self.client, self.limite, self.gastado, self.llamadas = client, limite_usd, 0.0, 0
        self.registro = []  # una fila por llamada: modelo, tokens y costo

    def crear(self, **kwargs):
        if self.gastado >= self.limite:
            raise RuntimeError(f"Límite de USD {self.limite:.2f} alcanzado en este notebook.")
        r = self.client.messages.create(**kwargs)
        p_in, p_out = PRECIOS.get(kwargs["model"], PRECIO_CONSERVADOR)
        costo = (r.usage.input_tokens * p_in + r.usage.output_tokens * p_out) / 1e6
        self.gastado += costo
        self.llamadas += 1
        self.registro.append({"modelo": kwargs["model"], "entrada": r.usage.input_tokens,
                              "salida": r.usage.output_tokens, "costo": costo})
        return r

    def __repr__(self):
        return f"{self.llamadas} llamadas · USD {self.gastado:.4f} de {self.limite:.2f}"

In [ ]:
import re, json, hashlib, difflib
from pathlib import Path
from typing import Literal
import pandas as pd
import pdfplumber
from pydantic import BaseModel, Field, ValidationError

DATA = Path("norma_mini_dataset")
SALIDAS = DATA / "salidas"
SALIDAS.mkdir(exist_ok=True)
MODELO = "claude-haiku-4-5"
api = Presupuesto(client, limite_usd=0.50)

# Chunks del Lab 1 (si no corriste el Lab 1, el dataset trae un respaldo con el mismo nombre)
chunks = [json.loads(l) for l in open(SALIDAS / "chunks_lab1.jsonl", encoding="utf-8")]
taxonomia = json.load(open(DATA / "taxonomia.json", encoding="utf-8"))
gold = pd.read_csv(DATA / "gold_criterios.csv")
print(len(chunks), "chunks ·", len(taxonomia), "nodos de taxonomía ·", api)

## 1 · El esquema: qué es un criterio para nosotros

Un criterio tiene cuatro campos. El campo `id_tema` solo admite los identificadores de la taxonomía: así el modelo no puede inventar temas.

### ✏️ Ejercicio 1
Completa el campo `articulos` del modelo `Criterio`: una lista de cadenas con una descripción que le diga al modelo el formato esperado (`"LFCE 58"`, `"CPEUM 28"`) y que, si no hay artículos, devuelva una lista vacía.

In [ ]:
TemaId = Literal[tuple(n["id"] for n in taxonomia)]

class Criterio(BaseModel):
    criterio: str = Field(description="Abstracción fiel del criterio en una oración, sin datos del caso concreto.")
    cita_literal: str = Field(description="Copia EXACTA del fragmento que sustenta el criterio, de 10 a 60 palabras. No cambies ni una palabra.")
    id_tema: TemaId = Field(description="Nodo más específico de la taxonomía.")
    # TODO: agrega el campo `articulos` (lista de str, vacía por defecto) con una descripción útil para el modelo

class Extraccion(BaseModel):
    criterios: list[Criterio] = Field(description="Criterios del fragmento. Lista vacía si no hay ninguno.")

In [ ]:
def esquema_plano(modelo):
    """JSON Schema de Pydantic con las referencias $ref resueltas en línea."""
    s = modelo.model_json_schema()
    defs = s.pop("$defs", {})
    def resolver(x):
        if isinstance(x, dict):
            if "$ref" in x:
                return resolver(defs[x["$ref"].split("/")[-1]])
            return {k: resolver(v) for k, v in x.items()}
        if isinstance(x, list):
            return [resolver(v) for v in x]
        return x
    return resolver(s)

ESQUEMA = esquema_plano(Extraccion)
assert "articulos" in ESQUEMA["properties"]["criterios"]["items"]["properties"], "Falta el campo articulos"
print("✅ Esquema listo")
print(json.dumps(ESQUEMA["properties"]["criterios"]["items"]["properties"]["id_tema"], ensure_ascii=False)[:300], "…")

## 2 · El prompt y la llamada

Le damos al modelo una definición operativa de criterio, una lista explícita de lo que **no** es criterio y la taxonomía. Para forzar el esquema usamos una *herramienta* con `tool_choice`: el modelo no puede responder con texto libre, solo llenando el formulario.

> Estos prompts son una versión simplificada para el curso. Los de producción de Norma+ son más largos y están separados por tipo de procedimiento.

In [ ]:
TAX_TEXTO = "\n".join(f"{n['id']}: {n['ruta_tema']}" for n in taxonomia)

SISTEMA = f"""Eres un analista jurídico especializado en competencia económica en México.
Recibirás un fragmento de una resolución. Extrae sus CRITERIOS JURÍDICOS: proposiciones generales sobre
cómo se interpreta o aplica una norma, que podrían invocarse en un caso distinto.

NO son criterios:
- hechos del caso (fechas, montos, porcentajes, conductas concretas);
- transcripciones de contratos, leyes u otros documentos;
- argumentos de las partes;
- la valoración de pruebas concretas;
- la aplicación de un criterio a los hechos del caso.

Reglas:
- La cita_literal debe copiarse EXACTAMENTE del fragmento.
- Si el fragmento no contiene criterios, devuelve una lista vacía.

Taxonomía (usa el id):
{TAX_TEXTO}"""

HERRAMIENTA = {"name": "registrar_criterios",
               "description": "Registra los criterios jurídicos extraídos del fragmento.",
               "input_schema": ESQUEMA}

In [ ]:
CACHE_RUTA = SALIDAS / "cache_lab2.json"
cache = json.load(open(CACHE_RUTA, encoding="utf-8")) if CACHE_RUTA.exists() else {}

def extraer(chunk):
    """Devuelve (Extraccion | None, error). Usa caché: el mismo chunk con el mismo prompt no se paga dos veces."""
    clave = hashlib.sha256((MODELO + SISTEMA + chunk["texto"]).encode()).hexdigest()
    if clave not in cache:
        r = api.crear(model=MODELO, max_tokens=1500, system=SISTEMA,
                      tools=[HERRAMIENTA], tool_choice={"type": "tool", "name": "registrar_criterios"},
                      messages=[{"role": "user", "content": f"Fragmento ({chunk['titulo']}):\n\n{chunk['texto']}"}])
        cache[clave] = next(b.input for b in r.content if b.type == "tool_use")
        json.dump(cache, open(CACHE_RUTA, "w", encoding="utf-8"), ensure_ascii=False)
    try:
        return Extraccion.model_validate(cache[clave]), None
    except ValidationError as e:
        return None, str(e)[:200]   # fallar cerrado: un esquema inválido no entra a la base

ejemplo = next(c for c in chunks if c["id_chunk"] == "R1-QUINTO")
resultado, error = extraer(ejemplo)
print(api)
for c in (resultado.criterios if resultado else []):
    print(f"\n[{c.id_tema}] {c.criterio}\n   cita: «{c.cita_literal}»\n   artículos: {c.articulos}")

## 3 · ¿La cita está realmente en el texto?

Los LLM a veces «mejoran» la cita: cambian una palabra, unen dos oraciones o la resumen. Para un abogado, una cita alterada es peor que ninguna. Por eso verificamos.

### ✏️ Ejercicio 2
Completa `es_literal`: normaliza ambos textos (espacios múltiples y saltos de línea a un solo espacio; comillas tipográficas a rectas) y devuelve `True` solo si la cita aparece completa dentro del chunk.

In [ ]:
def normalizar(s):
    # TODO: convierte comillas tipográficas (“ ” « ») a comillas rectas y colapsa espacios con re.sub
    return s

def es_literal(cita, texto):
    # TODO: True si la cita normalizada está contenida en el texto normalizado
    return False

In [ ]:
assert es_literal("basta  acreditar\nque", "…se consumó; basta acreditar que la conducta…")
assert not es_literal("basta probar que", "…se consumó; basta acreditar que la conducta…")
print("✅ Verificación literal correcta")

### Anclar cada cita a su página

El chunk dice entre qué páginas está, pero queremos la página exacta. Volvemos al PDF, quitamos el pie repetido (como en el Lab 1) y buscamos la cita. Si cruza un salto de página, la anclamos a la página donde empieza.

In [ ]:
def textos_por_pagina(ruta):
    with pdfplumber.open(ruta) as pdf:
        lineas = [[l["text"] for l in p.extract_text_lines()] for p in pdf.pages]
    firma = lambda t: re.sub(r"\d+", "#", t.strip())
    ultimas = [firma(ls[-1]) for ls in lineas if ls]
    pie = {f for f in ultimas if ultimas.count(f) >= 2}
    return [normalizar(" ".join(l for l in ls if firma(l) not in pie)) for ls in lineas]

PAGINAS = {p.name: textos_por_pagina(p) for p in sorted((DATA / "pdf").glob("*.pdf"))}

def anclar(documento, cita):
    c, pags = normalizar(cita), PAGINAS[documento]
    for i, t in enumerate(pags, 1):
        if c in t:
            return i
    for i in range(1, len(pags)):
        if c in pags[i - 1] + " " + pags[i]:
            return i
    return None

## 4 · Extraer todo el corpus

Recorremos los 20 chunks. Cada criterio recibe un **estado**:
- `verificado`: esquema válido y cita literal anclada a una página;
- `cita_no_literal`: el modelo alteró la cita; se aparta para revisión;
- `esquema_invalido`: la respuesta no respetó el esquema; el chunk entero se aparta.

In [ ]:
filas = []
for ch in chunks:
    res, err = extraer(ch)
    if res is None:
        filas.append({"id_chunk": ch["id_chunk"], "documento": ch["documento"], "estado": "esquema_invalido", "error": err})
        continue
    for c in res.criterios:
        literal = es_literal(c.cita_literal, ch["texto"])
        filas.append({"id_chunk": ch["id_chunk"], "documento": ch["documento"], "apartado": ch["titulo"],
                      **c.model_dump(),
                      "pagina": anclar(ch["documento"], c.cita_literal) if literal else None,
                      "estado": "verificado" if literal else "cita_no_literal"})

ext = pd.DataFrame(filas)
ext.to_json(SALIDAS / "criterios_lab2.jsonl", orient="records", lines=True, force_ascii=False)
print(api)
ext.estado.value_counts()

In [ ]:
# Las citas no literales, lado a lado con el fragmento más parecido del chunk
texto_chunk = {c["id_chunk"]: normalizar(c["texto"]) for c in chunks}
for _, f in ext[ext.estado == "cita_no_literal"].iterrows():
    t, c = texto_chunk[f.id_chunk], normalizar(f.cita_literal)
    m = difflib.SequenceMatcher(None, t, c).find_longest_match(0, len(t), 0, len(c))
    print(f"{f.id_chunk}\n  modelo: «{c}»\n  texto:  «{t[max(0, m.a - m.b): m.a - m.b + len(c)]}»\n")

## 5 · Un primer vistazo contra el gold

La evaluación formal llega en el Lab 4. Por ahora contamos cuántos criterios del gold encontró el modelo (una extracción coincide si su cita se traslapa con la cita del gold) y qué extrajo de más.

In [ ]:
def coincide(cita_a, cita_b, umbral=0.6):
    a, b = normalizar(cita_a).lower(), normalizar(cita_b).lower()
    if a in b or b in a:
        return True
    return difflib.SequenceMatcher(None, a.split(), b.split()).ratio() >= umbral

verif = ext[ext.estado == "verificado"]
gold["encontrado"] = [any(coincide(c, g) for c in verif[verif.documento == d].cita_literal)
                      for d, g in zip(gold.documento, gold.cita_literal)]
verif = verif.assign(en_gold=[any(coincide(c, g) for g in gold[gold.documento == d].cita_literal)
                              for d, c in zip(verif.documento, verif.cita_literal)])

print(f"Criterios del gold encontrados: {gold.encontrado.sum()} de {len(gold)}")
print(f"Extracciones verificadas que no están en el gold: {(~verif.en_gold).sum()}")
verif[~verif.en_gold][["id_chunk", "criterio", "cita_literal"]]

> **Para discutir:** revisa las extracciones que no están en el gold. ¿Son criterios que el gold omitió, o hechos y aplicaciones al caso que el modelo confundió? Esa pregunta es exactamente la que resuelve el **validador** del siguiente bloque. Y fíjate: una cita literal no garantiza que lo extraído sea un criterio; solo garantiza que existe.

## Cierre

| Lo que hicimos | Lo que en Norma+ es |
|---|---|
| Pydantic + herramienta forzada | Salidas estructuradas por esquema |
| `Literal` con los ids de la taxonomía | Clasificación restringida a nodos válidos |
| `es_literal` + `anclar` | Recuperación de evidencia: cita, página y caja |
| Estados `verificado` / `cita_no_literal` / `esquema_invalido` | Fallar cerrado |
| Caché por hash de modelo, prompt y chunk | Reproducibilidad y control de costos |

**Siguiente:** validación y fidelidad. ¿Lo extraído es un criterio o un hecho? ¿La abstracción dice lo mismo que la cita, o generaliza de más?